# CRA Tutorial — Hands-on AI on the National Research Platform

This notebook is the hands-on portion of the tutorial. Everything runs
**inside this JupyterHub session** — no separate pods to launch and no YAML to
apply. (YAML equivalents are kept as collapsible reveals so you can see the
Kubernetes version, but you never have to run them.)

A one-line vocabulary primer, in case these are new:

- **LLM** — a large language model (the thing that answers prompts).
- **Inference** — running a prompt through a model to get an answer.
- **RAG** — *retrieval-augmented generation*: look up relevant text first,
  then ask the model to answer **using only that text**.

**What you'll do**

1. Verify the session has the env vars, `kubectl`, and (optionally) a GPU.
2. Call the **NRP managed LLM** (`https://ellm.nrp-nautilus.io/v1`) from Python.
3. Run a **local LLM** (Ollama + `mistral`) on the session's GPU, side by side.
4. Build a **simple RAG pipeline** over hand-written facts using managed Milvus.
5. Build a **real RAG pipeline** over the NRP documentation.

When you're done, [`3_opencode.md`](3_opencode.md) points an agentic coding CLI
at this same managed LLM.

**How to run a notebook (skip if you've used Jupyter before)**

- Grey cells are code; this one is text. Click a cell and press
  **Shift + Enter** to run it. Run them **top to bottom, in order**.
- `[*]` means a cell is still running; `[7]` means it finished. If stuck, use
  **Kernel → Restart Kernel** and re-run from the top.
- Cells that print `Skipping — no GPU` are fine on a CPU-only session.

**Prerequisites**

- Spawn with **1 × NVIDIA-A10 GPU, ~4 CPU cores, and ~16 GB RAM** to run the
  local-Ollama half of sections 3 and 5 (the spawn form defaults to 1 core /
  8 GB — too low; the `mistral` load needs ~4 GB). The managed-LLM and RAG
  sections also work on a CPU-only session.
- The pod already exports `OPENAI_API_BASE` / `OPENAI_API_KEY` and has
  `kubectl` wired to the `nrp-training-k8s` namespace.


## 1. Setup check

Confirm the session has everything we'll need: the managed-LLM env vars,
`kubectl` against `nrp-training-k8s`, and `nvidia-smi` if you spawned with a
GPU. The Ollama sections will skip themselves gracefully if there's no GPU.


In [ ]:
import os, shutil, subprocess

print("OPENAI_API_BASE =", os.environ.get("OPENAI_API_BASE", "NOT SET"))
# Print the actual token. This is a shared workshop key — fine to show here;
# treat your own personal token as a secret.
key = os.environ.get("OPENAI_API_KEY", "")
print("OPENAI_API_KEY  =", key if key else "NOT SET")

kubectl = shutil.which("kubectl") or "/opt/conda/bin/kubectl"
print("kubectl path    =", kubectl)
print("kubectl version =", subprocess.run([kubectl, "version", "--client", "-o", "json"],
                                          capture_output=True, text=True).stdout[:120], "...")

print("can list pods in nrp-training-k8s:",
      subprocess.run([kubectl, "auth", "can-i", "list", "pods", "-n", "nrp-training-k8s"],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
# GPU detection. Sections 3 & 5 (local Ollama) need this; others are fine without.
import subprocess

HAS_GPU = False
try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True).stdout.strip()
    print("GPU(s) detected:")
    print(out)
    HAS_GPU = True
except (FileNotFoundError, subprocess.CalledProcessError):
    print("No GPU in this session pod.")
    print("Sections 3 & 5 (local Ollama) will skip the local-LLM half.")
    print("To run the full comparison, respawn with 1 x NVIDIA-A10 from the JupyterHub launcher.")


## 2. Option A — Use NRP's managed LLMs

There are **two ways to get an LLM in this notebook**, and they use the *same*
Python code:

- **Option A (this section): NRP's managed LLMs.** NRP runs a catalog of
  open-weights models for you behind one OpenAI-compatible URL. You don't run
  a pod and no GPU is billed to you — you just send HTTP requests.
- **Option B (section 3): run your own LLM locally** with Ollama, on the GPU
  attached to this JupyterHub session.

### What the NRP managed LLM service is

NRP hosts a **rotating catalog of open-weights LLMs**, reachable two ways:

- a **browser chat UI** (no setup, ChatGPT-like) at
  [`https://nrp-openwebui.nrp-nautilus.io`](https://nrp-openwebui.nrp-nautilus.io), and
- an **OpenAI-compatible API** at `https://ellm.nrp-nautilus.io/v1` — the one
  we use here.

You authenticate with a **bearer token**. Anyone in an LLM-enabled group can
mint one at [`https://nrp.ai/llmtoken`](https://nrp.ai/llmtoken); on this
training session it's already exported for you as `OPENAI_API_KEY`.

**The catalog rotates** as the open-source frontier moves. Roughly, today:

| Model | Size | Good for |
|---|---|---|
| `qwen3` | 397B | frontier reasoning, largest context |
| `gpt-oss` | 120B | agentic tool-calling / code |
| `gemma` | 31B | general chat, multimodal |
| `kimi`, `glm-5`, `minimax-m2` | 230B–1T | coding (under evaluation) |
| `qwen3-embedding` | 8B | embeddings only (not chat) |

> Some of these (`qwen3`, `minimax-m2`, `gpt-oss`, …) are **reasoning models** —
> they "think" before answering. We'll see what that means in code below.

### Why this matters: one API, many backends

The same `openai` Python SDK code targets:

- the **NRP managed endpoint** (this section),
- a **local Ollama / vLLM server** (section 3),
- or the OpenAI cloud — **just by changing `base_url`**.

That portability is the entire point of OpenAI-compatible APIs.

**NRP LLM docs:**
[overview](https://nrp.ai/documentation/userdocs/ai/llm-managed/) ·
[models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) ·
[chat interfaces](https://nrp.ai/documentation/userdocs/ai/llm-managed/chat-interfaces/) ·
[API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) ·
[client configs](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/)


### What the next five cells do

Each one does a single small thing, so you can see every moving part:

1. **List the catalog** — ask the endpoint which models it's serving right now.
2. **Define a `chat()` helper** — a tiny wrapper so every later call is one line.
3. **Ask one question** — a single request and response.
4. **Stream** — the same call, but printing tokens as they're generated.
5. **Swap the model** — identical code, just a different `model=` name.

Notice that the only NRP-specific thing is the `base_url`. Everything else is
the standard `openai` library you'd use against OpenAI itself.


In [ ]:
# List the models currently served by the NRP managed endpoint.
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
models = client.models.list()
print(f"{len(models.data)} models available right now:\n")
for m in sorted(models.data, key=lambda x: x.id):
    print(f"  {m.id}")


In [ ]:
# A tiny helper we'll reuse across this notebook.
#   - `max_tokens` is bounded so a runaway response can't hang the kernel.
#   - Reasoning models stream their chain-of-thought into a separate
#     `reasoning` field and only fill `content` once they conclude; if we ran
#     out of tokens mid-thought, we surface that reasoning instead of crashing.
def chat(prompt, model="gemma", system=None, max_tokens=512, llm=None):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = (llm or client).chat.completions.create(
        model=model, messages=msgs, max_tokens=max_tokens, temperature=0.2,
    )
    msg = resp.choices[0].message
    if msg.content:
        return msg.content
    reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    if reasoning:
        return f"(no final answer within max_tokens={max_tokens}; reasoning so far:)\n{reasoning}"
    return "(model produced no content — try raising max_tokens or another model)"


In [ ]:
# Single chat completion.
print(chat(
    "What is the National Research Platform?",
    system="Answer in one sentence.",
))


In [ ]:
# Streaming — tokens arrive as they're generated.
stream = client.chat.completions.create(
    model="gemma",
    messages=[{"role": "user", "content": "Write a haiku about GPUs."}],
    stream=True,
    max_tokens=80,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


### Standard vs. reasoning models

Most models answer immediately. **Reasoning models** (`qwen3`, `minimax-m2`,
`gpt-oss`, …) first write a private chain-of-thought into a separate
`reasoning` field, then put the final answer in `content`. The practical
catch: that thinking spends tokens, so you must give them a **larger
`max_tokens`** or they'll hit the limit before producing any `content` — which
is exactly the empty-answer case the `chat()` helper guards against.


In [ ]:
# Switch to a smaller model — same code, just change the `model` arg.
print(chat("Explain Kubernetes namespaces in two sentences.", model="gemma-small"))

# Now a reasoning model. We give it room (max_tokens=800) so it finishes
# thinking AND produces a final answer — with too few tokens the whole budget
# is spent reasoning and `content` comes back empty (what the helper guards).
print("\n--- minimax-m2 (a reasoning model) ---")
print(chat("In one sentence, name a strength of this model.", model="minimax-m2", max_tokens=800))

# Try the others yourself — e.g. model="gpt-oss" (fast) or model="qwen3"
# (the 397B flagship; highest quality but slowest to respond).


<details>
<summary><b>What this would look like as a Kubernetes pod (click to expand)</b></summary>

If you wanted to run your *own* LLM server on NRP instead of using the managed
endpoint, you'd request a GPU pod that boots an OpenAI-compatible server like
vLLM or HuggingFace TGI. The pod below would expose `/v1/chat/completions` on
the same port, so the Python code above works against it unchanged — only the
`base_url` changes (e.g., to `http://127.0.0.1:8080/v1` after a port-forward).

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: tutorial-<username>-tgi
  namespace: nrp-training-k8s
spec:
  containers:
  - name: tgi
    image: ghcr.io/huggingface/text-generation-inference:2.1.1
    args: ["--model-id", "HuggingFaceH4/zephyr-7b-beta"]
    resources:
      requests: { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
      limits:   { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
        - matchExpressions:
          - { key: nvidia.com/gpu.product, operator: In, values: [NVIDIA-A10] }
  tolerations:
  - { key: nautilus.io/reservation, operator: Equal, value: nrp, effect: NoSchedule }
```

Compared to the managed endpoint:

| | Managed (`ellm.nrp-nautilus.io`) | Self-hosted pod (YAML above) |
|---|---|---|
| Setup | none — just an API call | `kubectl apply` + wait for pull + port-forward |
| Cost | shared, no GPU billed to you | a GPU on the workshop reservation |
| Control | pick from the catalog | pick *any* HF model, quantization, runtime flags |
| Use when | classroom demos, prototypes | research that needs a specific model or config |

</details>


## 3. Option B — Run your own LLM locally (Ollama on the session GPU)

Same OpenAI-compatible API, but now the model runs on **your** GPU inside this
JupyterHub session instead of on NRP's shared endpoint. We install Ollama,
pull `mistral` (7B Q4, ~4 GB), and hit it at `http://localhost:11434/v1` — then
ask the same prompt against NRP's managed `gemma` and our local `mistral` side
by side. **Only `base_url` changes between the two.**

> These cells skip themselves if `HAS_GPU` is `False`. Respawn with
> **1 × NVIDIA-A10** to run the local half.

> **Heads-up on speed:** Ollama and its models install to fast local `/tmp`
> (re-downloaded each session, ~10 s) so the GPU is detected reliably. The
> first `mistral` call **pulls ~4 GB**, which takes a minute or two; after that
> it's loaded on the A10 and responds in a second or two.


In [ ]:
# Install Ollama onto the session's FAST local disk (/tmp), NOT the home
# directory. Ollama ships ~3.5 GB of CUDA libraries; the home volume is
# networked storage (Ceph), and reading 3.5 GB from it is too slow for Ollama's
# GPU probe — it times out and silently falls back to CPU. /tmp is node-local
# and fast, so the A10 is detected in under a second. (/tmp is wiped when the
# session stops, so this re-downloads each session — it only takes ~10 s.)
import os, shutil, subprocess

OLLAMA_DIR = "/tmp/ol"
OLLAMA_BIN = f"{OLLAMA_DIR}/bin/ollama"
OLLAMA_VERSION = "v0.24.0"  # pinned to a known-good release

if not HAS_GPU:
    print("Skipping Ollama install — no GPU in this session.")
elif os.path.exists(OLLAMA_BIN):
    print("Ollama already installed at", OLLAMA_BIN)
else:
    print(f"Downloading Ollama {OLLAMA_VERSION} to {OLLAMA_DIR} (~10 s)...")
    os.makedirs(OLLAMA_DIR, exist_ok=True)
    url = f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
    subprocess.run(
        f"curl -fsSL {url} | tar --use-compress-program=unzstd -xf - -C {OLLAMA_DIR}",
        shell=True, check=True,
    )
if HAS_GPU:
    os.environ["PATH"] = f"{OLLAMA_DIR}/bin" + os.pathsep + os.environ["PATH"]
    print("ollama binary at:", shutil.which("ollama"))


In [ ]:
# Start `ollama serve` in the background and wait until it answers.
#   - Everything (binary, CUDA libs, models) lives under /tmp/ol on fast local
#     disk, so the GPU probe finds the A10 in well under a second.
#   - OLLAMA_LLM_LIBRARY + LD_LIBRARY_PATH point at the bundled CUDA libraries.
#   - OLLAMA_MODELS keeps pulled models on /tmp too, so they load fast.
import os, socket, subprocess, time, urllib.request, urllib.error

def ollama_alive():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return True
    except (urllib.error.URLError, ConnectionResetError, OSError, socket.timeout, TimeoutError):
        return False

LIB = f"{OLLAMA_DIR}/lib/ollama"

if not HAS_GPU:
    print("Skipping Ollama serve — no GPU in this session.")
elif ollama_alive():
    print("Ollama already running.")
else:
    print("Starting `ollama serve` in the background...")
    ollama_env = {
        **os.environ,
        "OLLAMA_HOST": "0.0.0.0:11434",
        "OLLAMA_LOAD_TIMEOUT": "15m",
        "OLLAMA_MODELS": f"{OLLAMA_DIR}/models",
        "OLLAMA_LIBRARY_PATH": LIB,
        "OLLAMA_LLM_LIBRARY": "cuda_v12",
        "LD_LIBRARY_PATH": ":".join([
            LIB, LIB + "/cuda_v12", "/usr/lib/x86_64-linux-gnu",
            os.environ.get("LD_LIBRARY_PATH", ""),
        ]),
    }
    subprocess.Popen(f"nohup {OLLAMA_BIN} serve > /tmp/ollama.log 2>&1 &", shell=True, env=ollama_env)
    t0 = time.time(); deadline = t0 + 120
    while time.time() < deadline:
        time.sleep(2)
        if ollama_alive():
            print(f"Ollama up after ~{int(time.time() - t0)}s.")
            break
    else:
        print("Ollama did not come up within 120s. Check /tmp/ollama.log")

# Report which device Ollama is using.
if HAS_GPU and ollama_alive():
    log = subprocess.run("grep -iE 'inference compute' /tmp/ollama.log | tail -1",
                         shell=True, capture_output=True, text=True).stdout.lower()
    if "library=cuda" in log:
        print("Device: GPU (CUDA) — the A10 was detected. \U0001f389")
    elif "id=cpu" in log:
        print("Device: CPU — GPU probe fell back to CPU; check /tmp/ollama.log")
    else:
        print("Device: unknown — see /tmp/ollama.log")


In [ ]:
# Pull the mistral 7B model (~4 GB). First time only.
import subprocess, time, json, urllib.request

def already_pulled(name="mistral"):
    try:
        r = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2))
        return any(name in m["name"] for m in r.get("models", []))
    except Exception:
        return False

if not HAS_GPU:
    print("Skipping — no GPU.")
elif already_pulled("mistral"):
    print("mistral already pulled.")
else:
    print("Pulling mistral... (~4 GB, 2-4 minutes the first time)")
    # Drive the pull via the HTTP API instead of `ollama pull`, because the CLI
    # writes \r-based progress bars that buffer indefinitely under nbconvert.
    body = json.dumps({"name": "mistral", "stream": True}).encode()
    req  = urllib.request.Request("http://localhost:11434/api/pull", data=body,
                                  headers={"Content-Type": "application/json"})
    t0 = time.time()
    last_status = None
    with urllib.request.urlopen(req) as resp:
        for raw in resp:
            try:
                evt = json.loads(raw.decode())
            except Exception:
                continue
            status = evt.get("status", "")
            # Only print on status transitions (not every kilobyte) to keep output tidy
            if status and status != last_status:
                print(f"  [{int(time.time()-t0):3d}s] {status}")
                last_status = status
            if evt.get("error"):
                raise RuntimeError(evt["error"])
    print(f"Done in {int(time.time()-t0)}s.")


In [ ]:
# Hello-world against the local model. Same OpenAI client, different base_url.
# Heads up: the first call after `ollama serve` started will be slow (~30-120s
# while llama.cpp loads the 4 GB of weights into VRAM). Subsequent calls in
# this notebook reuse the loaded model and respond in 1-2 seconds.
from openai import OpenAI

if HAS_GPU:
    local = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1",
                   timeout=900)  # generous timeout for the cold load
    print(chat(
        "Say hi from the GPU in this JupyterHub session.",
        model="mistral", llm=local, max_tokens=80,
    ))
else:
    print("Skipping — no GPU.")


In [ ]:
# Side-by-side: same prompt, managed gemma vs local mistral.
prompt = "Explain in two sentences why a research cluster might use Kubernetes namespaces."

print("=" * 78)
print("NRP MANAGED (gemma)")
print("=" * 78)
print(chat(prompt, model="gemma"))

if HAS_GPU:
    print()
    print("=" * 78)
    print("LOCAL (mistral, on this pod's GPU)")
    print("=" * 78)
    print(chat(prompt, model="mistral", llm=local))
else:
    print("\n(Local half skipped — no GPU.)")


**What to notice.** The Python code is byte-for-byte identical — same SDK,
same `chat.completions.create`, same messages. Only `base_url` differs.
Latency, answer style, and request privacy will differ:

| | Managed `gemma` | Local `mistral` (this pod) |
|---|---|---|
| Where it runs | NRP GPUs you don't see | the A10 attached to this pod |
| Latency to first token | ~1.5 s | ~6 s (warm) |
| GPU billed | none | yours, while pod runs |
| Privacy | request leaves your namespace | never leaves your pod |
| Catalog | the NRP model list | anything `ollama pull` supports |

<details>
<summary><b>What the Ollama equivalent looks like as a YAML pod (click to expand)</b></summary>

If you ran Ollama in its own dedicated pod instead of inside the JupyterHub
session (e.g., to share one Ollama server across multiple notebook sessions,
or to keep model weights on a separate PVC), it would look like this:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: tutorial-<username>-ollama
  namespace: nrp-training-k8s
spec:
  containers:
  - name: ollama
    image: ollama/ollama:0.2.8
    resources:
      requests: { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
      limits:   { cpu: "4", memory: 16Gi, nvidia.com/gpu: 1 }
    ports:
    - { containerPort: 11434 }
  affinity:
    nodeAffinity:
      requiredDuringSchedulingIgnoredDuringExecution:
        nodeSelectorTerms:
        - matchExpressions:
          - { key: nvidia.com/gpu.product, operator: In, values: [NVIDIA-A10] }
  tolerations:
  - { key: nautilus.io/reservation, operator: Equal, value: nrp, effect: NoSchedule }
```

You'd then `kubectl port-forward pod/tutorial-<username>-ollama 11434:11434`
and point your notebook at `http://localhost:11434/v1` — same code, same
result. The in-notebook path you just ran is the same thing, minus the YAML.

</details>


## 4. Simple RAG with managed Milvus

RAG ("retrieval-augmented generation") is the standard pattern for asking an
LLM about facts the model wasn't trained on: you (1) embed your corpus into a
vector database, (2) retrieve the chunks closest to the user's question, and
(3) include those chunks in the prompt with a "answer only from this context"
system message.

NRP runs a managed [Milvus](https://milvus.io) at `milvus.nrp-nautilus.io:50051`.
The credentials live in a Kubernetes Secret in our namespace, which we'll
load via `kubectl` (one shell-out, then it's just Python).

This first RAG pass uses six hand-written facts so you can see every moving
part. Section 5 swaps the corpus for the actual NRP documentation.


### The five mechanical steps of RAG

The cells below walk through RAG one step at a time:

1. **Load credentials** for the vector database (Milvus) from a Kubernetes Secret.
2. **Create a collection** — a table that stores text plus its embedding vector.
3. **Embed + insert** your documents (turn each chunk of text into a vector).
4. **Retrieve** — embed the *question* and find the closest chunks.
5. **Ask** — hand those chunks to the LLM with "answer only from this context."

An **embedding** is just a list of numbers that captures a piece of text's
meaning; chunks about the same topic land near each other, which is what makes
"find the closest chunks" work.


In [ ]:
# Pull Milvus creds from the namespace Secret. The kubeconfig in this pod
# already points at the jupyterhub-sa service account which has read on this Secret.
import os, base64, subprocess

def k_secret(key):
    out = subprocess.check_output([
        "kubectl", "get", "secret", "nrp-training-milvus-credentials",
        "-n", "nrp-training-k8s", "-o", f"jsonpath={{.data.{key}}}",
    ])
    return base64.b64decode(out).decode()

os.environ["MILVUS_HOST"]     = k_secret("host")
os.environ["MILVUS_PORT"]     = k_secret("port")
os.environ["MILVUS_USER"]     = k_secret("username")
os.environ["MILVUS_PASSWORD"] = k_secret("password")
os.environ["MILVUS_SECURE"]   = k_secret("secure")
os.environ["MILVUS_DB_NAME"]  = k_secret("database")

print(f"Milvus: {os.environ['MILVUS_USER']}@{os.environ['MILVUS_HOST']}:{os.environ['MILVUS_PORT']}")
print(f"  database = {os.environ['MILVUS_DB_NAME']}")
print(f"  secure   = {os.environ['MILVUS_SECURE']}")


In [ ]:
# Install RAG dependencies. The PyTorch image already has torch + transformers,
# so we just add pymilvus, sentence-transformers, and the text splitter.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymilvus>=2.3", "sentence-transformers>=2.2",
                "langchain-text-splitters>=0.0.1"], check=True)
print("Dependencies installed.")


In [ ]:
# Connect to Milvus and create a per-user collection for this demo.
import os, getpass, re
from pymilvus import (
    connections, utility, Collection, CollectionSchema, FieldSchema, DataType,
)

connections.connect(
    alias="default",
    host=os.environ["MILVUS_HOST"], port=os.environ["MILVUS_PORT"],
    user=os.environ["MILVUS_USER"], password=os.environ["MILVUS_PASSWORD"],
    secure=os.environ["MILVUS_SECURE"].lower() == "true",
    db_name=os.environ["MILVUS_DB_NAME"],
)

# Per-user collection so demos don't collide.
user_slug = re.sub(r"[^a-z0-9_]+", "_", getpass.getuser().lower())[:32]
SIMPLE_COLLECTION = f"simple_rag_demo_{user_slug}"

if utility.has_collection(SIMPLE_COLLECTION):
    print(f"Dropping existing collection {SIMPLE_COLLECTION} for a clean demo run...")
    utility.drop_collection(SIMPLE_COLLECTION)

schema = CollectionSchema(fields=[
    FieldSchema(name="pk",     dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=128),
    FieldSchema(name="text",   dtype=DataType.VARCHAR, max_length=4000),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=384),
])
coll = Collection(name=SIMPLE_COLLECTION, schema=schema,
                  description="AI Unlocked simple RAG demo corpus")
coll.create_index("vector", {"metric_type": "COSINE", "index_type": "AUTOINDEX", "params": {}})
print(f"Created collection: {SIMPLE_COLLECTION}")


In [ ]:
# Six hand-written facts about NRP. In a real workflow you'd point this at a
# document folder; we keep it tiny so you can read the whole corpus.
DOCS = [
    ("nrp-overview",
     "The National Research Platform (NRP) is a shared national cyberinfrastructure "
     "built on the Nautilus Kubernetes cluster. It provides hundreds of GPU nodes, "
     "shared storage, and managed services like JupyterHub and a managed LLM endpoint."),
    ("nrp-gpus",
     "NRP exposes GPUs through Kubernetes resource keys. Generic GPUs use "
     "nvidia.com/gpu, specific products use keys like nvidia.com/a100 or "
     "nvidia.com/h200. Pods can also constrain themselves to a model with "
     "nvidia.com/gpu.product node-affinity."),
    ("nrp-managed-llm",
     "NRP runs a managed LLM service at https://ellm.nrp-nautilus.io/v1. It is "
     "OpenAI-compatible. Users authenticate with a bearer token minted at "
     "https://nrp.ai/llmtoken. The catalog rotates and includes qwen3, gpt-oss, "
     "gemma, and minimax-m2 among others."),
    ("nrp-milvus",
     "NRP runs a managed Milvus vector database reachable at "
     "milvus.nrp-nautilus.io:50051 (gRPC, TLS on). Per-user credentials are "
     "minted at https://nrp.ai/milvus."),
    ("nrp-policy-limits",
     "All workloads on NRP must declare CPU and memory requests and limits. A "
     "Gatekeeper admission policy rejects pods that omit them. The pattern used "
     "in workshop manifests is requests == limits."),
    ("nrp-policy-sleep",
     "Pods that idle indefinitely (e.g., `sleep infinity` in a Job) are against "
     "NRP policy and will result in the user being banned from the cluster. "
     "Always run real workloads or batch jobs with a defined exit."),
]
print(f"Loaded {len(DOCS)} hand-written facts.")


In [ ]:
# Embed with all-MiniLM-L6-v2 (384 dim, runs fine on CPU) and insert.
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
sources = [s for s, _ in DOCS]
texts   = [t for _, t in DOCS]
vectors = embedder.encode(texts, normalize_embeddings=True).tolist()

coll.insert([sources, texts, vectors])
coll.flush()
coll.load()
print(f"Inserted {len(DOCS)} chunks; collection now has {coll.num_entities} entities.")


In [ ]:
# Helper: retrieve top-k chunks for a question.
def retrieve(question, k=3):
    qvec = embedder.encode([question], normalize_embeddings=True).tolist()
    hits = coll.search(
        data=qvec, anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": 64}},
        limit=k, output_fields=["source", "text"],
    )
    return [(h.entity.get("source"), h.entity.get("text"), h.distance) for h in hits[0]]

# Helper: answer a question by RAG against `llm_client` (managed or local).
SYSTEM = (
    "You are a NRP support assistant. Answer the question using ONLY the context "
    "provided. If the context does not contain the answer, say so explicitly. "
    "Cite the source labels you used (e.g., [nrp-gpus])."
)
def ask(question, llm_client, model, max_tokens=400):
    hits = retrieve(question, k=3)
    context = "\n\n".join(f"[{src}] {txt}" for src, txt, _ in hits)
    resp = llm_client.chat.completions.create(
        model=model, temperature=0.1, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": f"Question: {question}\n\nContext:\n{context}"},
        ],
    )
    content = resp.choices[0].message.content or "(model produced no `content`)"
    return content, hits

QUESTION = "What happens to a pod on NRP that has no CPU or memory limits?"
print("Retrieved chunks:")
for src, _, score in retrieve(QUESTION, k=3):
    print(f"  [{src}]  score={score:.3f}")


In [ ]:
# Same question, managed gemma.
ans, hits = ask(QUESTION, client, "gemma")
print(ans)


In [ ]:
# Same question, local mistral on the session GPU.
if HAS_GPU:
    ans, hits = ask(QUESTION, local, "mistral")
    print(ans)
else:
    print("(Skipped — no GPU.)")


**What to notice.** Both backends see the same retrieved chunks (same Milvus
collection, same query embedding). Differences in the answers come purely
from the model itself:

- The managed `gemma` typically sticks to the context and cites the source.
- Smaller local models like `mistral` 7B sometimes fall back on parametric
  knowledge and add facts that aren't in the chunks — even when told not to.
  This is exactly the kind of behavior a docs-grounded assistant has to
  guard against, and trying a heavier local model (`llama3.1:70b`,
  `qwen2.5:32b`) usually makes the difference shrink.


## 5. Real RAG over the NRP documentation

Same pipeline, real corpus. We sparse-clone the public `nrp-site` repo
(Astro/Starlight Markdown source for the docs at `https://nrp.ai/documentation`),
chunk every page, embed, and store in the shared `nrp_docs_rag` collection.

The collection persists across runs — if someone in the workshop already
indexed today, the embed/insert step is skipped and we go straight to Q&A.


In [ ]:
# Sparse-clone just the documentation subtree.
import os, subprocess, shutil
from pathlib import Path

REPO_URL = "https://gitlab.nrp-nautilus.io/prp/nrp-site.git"
REPO_DIR = Path("/tmp/nrp-site")
REPO_SUBDIR = "src/content/docs/Documentation"

if (REPO_DIR / REPO_SUBDIR).exists():
    print(f"Repo already cloned at {REPO_DIR}. Updating with git pull...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "--quiet"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"Sparse-cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run([
        "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
        "--quiet", REPO_URL, str(REPO_DIR),
    ], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set", REPO_SUBDIR], check=True)

files = sorted(p for p in (REPO_DIR / REPO_SUBDIR).rglob("*") if p.suffix in (".md", ".mdx"))
print(f"Found {len(files)} markdown files.")


In [ ]:
# Strip Astro/Starlight chrome and reconstruct citable URLs from filesystem paths.
import re
URL_BASE = "https://nrp.ai/documentation"

_FRONTMATTER_RE     = re.compile(r"\A---\n.*?\n---\n", re.DOTALL)
_IMPORT_RE          = re.compile(r"^\s*import\s+.+?from\s+['\"][^'\"]+['\"]\s*;?\s*$", re.MULTILINE)
_ADMONITION_OPEN_RE = re.compile(r":::\w+(?:\[[^\]]*\])?\s*\n")
_ADMONITION_CLOSE_RE= re.compile(r"^:::\s*$", re.MULTILINE)

def strip_mdx(text):
    text = _FRONTMATTER_RE.sub("", text, count=1)
    text = _IMPORT_RE.sub("", text)
    text = _ADMONITION_OPEN_RE.sub("", text)
    text = _ADMONITION_CLOSE_RE.sub("", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def path_to_url(p):
    rel = p.relative_to(REPO_DIR / REPO_SUBDIR)
    parts = list(rel.parts)
    parts[-1] = re.sub(r"\.(md|mdx)$", "", parts[-1])
    if parts[-1].lower() == "index":
        parts = parts[:-1]
    return f"{URL_BASE}/{'/'.join(parts)}".rstrip("/")

pages = []
for p in files:
    try:
        raw = p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    text = strip_mdx(raw)
    if len(text) < 80:
        continue
    pages.append((path_to_url(p), text))
print(f"Kept {len(pages)} pages after stripping chrome + length filter.")


In [ ]:
# Chunk every page into ~900-char pieces with 120-char overlap.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = []
for url, text in pages:
    for piece in splitter.split_text(text):
        chunks.append((url, piece))
print(f"Chunked into {len(chunks)} pieces.")


In [ ]:
# Create (or reuse) the shared nrp_docs_rag collection.
DOCS_COLLECTION = "nrp_docs_rag"

if utility.has_collection(DOCS_COLLECTION):
    docs_coll = Collection(DOCS_COLLECTION)
    docs_coll.load()
    print(f"Reusing existing collection {DOCS_COLLECTION} with {docs_coll.num_entities} entities.")
    REBUILD_NEEDED = docs_coll.num_entities == 0
else:
    schema = CollectionSchema(fields=[
        FieldSchema(name="pk",     dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="url",    dtype=DataType.VARCHAR, max_length=512),
        FieldSchema(name="text",   dtype=DataType.VARCHAR, max_length=4000),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=384),
    ])
    docs_coll = Collection(name=DOCS_COLLECTION, schema=schema,
                           description="NRP documentation RAG corpus (sparse clone of nrp-site)")
    docs_coll.create_index("vector", {"metric_type": "COSINE", "index_type": "AUTOINDEX", "params": {}})
    print(f"Created collection {DOCS_COLLECTION}.")
    REBUILD_NEEDED = True


In [ ]:
# Embed + insert. Skipped automatically if the collection is already populated.
if REBUILD_NEEDED:
    print(f"Embedding + inserting {len(chunks)} chunks (CPU embedding, ~30-60s)...")
    BATCH = 64
    for i in range(0, len(chunks), BATCH):
        batch  = chunks[i:i+BATCH]
        urls   = [u for u, _ in batch]
        texts  = [t[:3990] for _, t in batch]
        vecs   = embedder.encode(texts, normalize_embeddings=True).tolist()
        docs_coll.insert([urls, texts, vecs])
        if (i // BATCH) % 4 == 0 or i + BATCH >= len(chunks):
            print(f"  inserted {min(i+BATCH, len(chunks))}/{len(chunks)}")
    docs_coll.flush()
    docs_coll.load()
    print(f"Done. Collection has {docs_coll.num_entities} entities.")
else:
    print("Collection already populated — skipping embed + insert.")


In [ ]:
# Retrieve + answer over the docs collection.
def retrieve_docs(question, k=4):
    qvec = embedder.encode([question], normalize_embeddings=True).tolist()
    hits = docs_coll.search(
        data=qvec, anns_field="vector",
        param={"metric_type": "COSINE", "params": {"ef": 64}},
        limit=k, output_fields=["url", "text"],
    )
    return [(h.entity.get("url"), h.entity.get("text"), h.distance) for h in hits[0]]

DOCS_SYSTEM = (
    "You are an assistant for NRP Nautilus users. Answer the question using ONLY "
    "the context provided. If the context does not contain the answer, say so. "
    "Cite the URLs you used."
)
def ask_docs(question, llm_client, model, max_tokens=500):
    hits = retrieve_docs(question, k=4)
    context = "\n\n".join(f"Source: {url}\n{txt}" for url, txt, _ in hits)
    resp = llm_client.chat.completions.create(
        model=model, temperature=0.1, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": DOCS_SYSTEM},
            {"role": "user",   "content": f"Question: {question}\n\nContext:\n{context}"},
        ],
    )
    content = resp.choices[0].message.content or "(model produced no `content`)"
    return content, hits

Q1 = "Is it okay to run `sleep infinity` in a Job on NRP?"
Q2 = "How do I expose a web service over HTTPS on NRP?"


In [ ]:
# Q1 via managed gemma.
ans, hits = ask_docs(Q1, client, "gemma")
print(f"Q: {Q1}\n")
print(ans)
print("\nSources:")
for url, _, score in hits:
    print(f"  - {url}  (score={score:.3f})")


In [ ]:
# Q1 via local mistral.
if HAS_GPU:
    ans, hits = ask_docs(Q1, local, "mistral")
    print(f"Q: {Q1}\n")
    print(ans)
    print("\nSources:")
    for url, _, score in hits:
        print(f"  - {url}  (score={score:.3f})")
else:
    print("(Skipped — no GPU.)")


In [ ]:
# Q2 via managed gemma.
ans, hits = ask_docs(Q2, client, "gemma")
print(f"Q: {Q2}\n")
print(ans)
print("\nSources:")
for url, _, score in hits:
    print(f"  - {url}  (score={score:.3f})")


In [ ]:
# Q2 via local mistral.
if HAS_GPU:
    ans, hits = ask_docs(Q2, local, "mistral")
    print(f"Q: {Q2}\n")
    print(ans)
    print("\nSources:")
    for url, _, score in hits:
        print(f"  - {url}  (score={score:.3f})")
else:
    print("(Skipped — no GPU.)")


**What to notice.** Real NRP policy questions get cited answers grounded in
the docs you can `curl` yourself. Same caveats as section 4 — small local
models will sometimes embellish; the managed catalog's larger models stick
to context more reliably. Swap `model="gemma"` for `"qwen3"` or `"gpt-oss"`
to see how a different managed model behaves on the same retrieved chunks.

<details>
<summary><b>What this whole pipeline looks like as a Kubernetes pod (click to expand)</b></summary>

If you wanted the entire RAG stack — Ollama + Milvus client + the indexer
script — as a long-running pod (e.g., a shared service for a class), you'd
write something like:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: tutorial-<username>-rag
  namespace: nrp-training-k8s
spec:
  containers:
  - name: rag
    image: nvidia/cuda:12.1.0-runtime-ubuntu22.04
    resources:
      requests: { cpu: "4", memory: 24Gi, nvidia.com/gpu: 1 }
      limits:   { cpu: "4", memory: 24Gi, nvidia.com/gpu: 1 }
    env:
    - { name: MILVUS_HOST,     valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: host } } }
    - { name: MILVUS_PORT,     valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: port } } }
    - { name: MILVUS_USER,     valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: username } } }
    - { name: MILVUS_PASSWORD, valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: password } } }
    - { name: MILVUS_SECURE,   valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: secure } } }
    - { name: MILVUS_DB_NAME,  valueFrom: { secretKeyRef: { name: nrp-training-milvus-credentials, key: database } } }
    - { name: OPENAI_API_BASE, valueFrom: { secretKeyRef: { name: nrp-llm-token, key: OPENAI_API_BASE } } }
    - { name: OPENAI_API_KEY,  valueFrom: { secretKeyRef: { name: nrp-llm-token, key: OPENAI_API_KEY } } }
```

The pod above is exactly what `yamls/milvus-rag.yaml` does. The notebook
you just ran is the same pipeline, condensed: env vars come from `kubectl get
secret` instead of `secretKeyRef`, and the GPU is the one already attached
to your JupyterHub session.

</details>


## 6. Cleanup

The simple-RAG collection is per-user and demo-only — drop it. The
`nrp_docs_rag` collection is shared and stays. Stopping the local Ollama
process frees the GPU memory for other notebook cells.


In [ ]:
# Drop the per-user simple-RAG collection.
if utility.has_collection(SIMPLE_COLLECTION):
    utility.drop_collection(SIMPLE_COLLECTION)
    print(f"Dropped {SIMPLE_COLLECTION}.")
else:
    print(f"{SIMPLE_COLLECTION} already gone.")


In [ ]:
# Stop the local Ollama daemon (frees GPU memory).
import subprocess
subprocess.run(["pkill", "-x", "ollama"], check=False)
print("Ollama stopped.")


## Takeaways

- The same OpenAI-compatible Python code targeted three different inference
  backends: NRP's managed `ellm.nrp-nautilus.io`, a local Ollama on this
  session's GPU, and (via the collapsible YAML reveals) self-hosted TGI/vLLM
  pods. Only `base_url` changed.
- Managed LLMs are the lowest-friction path for classrooms — no token
  handoff, no GPU billed to students. Self-hosted pods make sense when you
  need a specific model, quantization, or runtime control.
- RAG is "embed your docs, retrieve top-k for the question, prompt the LLM
  with 'answer from context only.'" Vector backend (Milvus), embedding model
  (`all-MiniLM-L6-v2`), and LLM are all independently swappable.
- Whatever you do in this notebook, the YAML equivalent is one
  `kubectl apply` away — useful when you outgrow the JupyterHub session.

**Next:** [`3_opencode.md`](3_opencode.md) — point an agentic coding CLI
(`opencode`) at this same managed LLM and have it write a small program.
